In [ ]:
import os
import numpy as np
from PIL import Image

import random
import torch
import torch.nn.functional as F
from torchvision.utils import save_image
import torch.optim as optim

from torchvision import transforms
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

from diffusers import StableDiffusionInpaintPipeline, AutoencoderKL
import timm
import lpips
import sys
import glob

sys.path.append('/root/StableGuard/locmark') # instead of the top directory
from helper import load_images_from_path, norm_imagenet, denorm_imagenet, load_image, load_mask
from locmark import LocMark

val_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    # transforms.CenterCrop(224),
    transforms.ToTensor(),
    # normalize_img,
])

def norm_tensor(tensor):
    t = tensor.clone().detach()
    
    min_val = t.min()
    max_val = t.max()

    tensor_norm = (tensor - min_val) / (max_val - min_val)

    print(f"Tensor normalized: min={tensor_norm.min()}, max={tensor_norm.max()}")
    
    return tensor_norm, min_val, max_val

def denorm_tensor(tensor, original_min=None, original_max=None):
    t = tensor.clone().detach()

    return t * (original_max - original_min) + original_min

def create_random_mask(img_pt, num_masks=1, mask_percentage=0.1, max_attempts=100):
    _, _, height, width = img_pt.shape
    mask_area = int(height * width * mask_percentage)
    masks = torch.zeros((num_masks, 1, height, width), dtype=img_pt.dtype)

    if mask_percentage >= 0.999:
        # Full mask for entire image
        return torch.ones((num_masks, 1, height, width), dtype=img_pt.dtype).to(img_pt.device)

    for ii in range(num_masks):
        placed = False
        attempts = 0
        while not placed and attempts < max_attempts:
            attempts += 1

            max_dim = int(mask_area ** 0.5)
            mask_width = random.randint(1, max_dim)
            mask_height = mask_area // mask_width

            # Allow broader aspect ratios for larger masks
            aspect_ratio = mask_width / mask_height if mask_height != 0 else 0
            if 0.25 <= aspect_ratio <= 4:  # Looser ratio constraint
                if mask_height <= height and mask_width <= width:
                    x_start = random.randint(0, width - mask_width)
                    y_start = random.randint(0, height - mask_height)
                    overlap = False
                    for jj in range(ii):
                        if torch.sum(masks[jj, :, y_start:y_start + mask_height, x_start:x_start + mask_width]) > 0:
                            overlap = True
                            break
                    if not overlap:
                        masks[ii, :, y_start:y_start + mask_height, x_start:x_start + mask_width] = 1
                        placed = True

        if not placed:
            # Fallback: just fill a central region if all attempts fail
            print(f"Warning: Failed to place mask {ii}, using fallback.")
            center_h = height // 2
            center_w = width // 2
            half_area = int((mask_area // 2) ** 0.5)
            h_half = min(center_h, half_area)
            w_half = min(center_w, half_area)
            masks[ii, :, center_h - h_half:center_h + h_half, center_w - w_half:center_w + w_half] = 1

    return masks.to(img_pt.device)

In [ ]:
class Params:
    """Hyperparameters and configuration settings for LocMark."""
    def __init__(self):
        # --- System & Paths ---
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.image_datasets = '/mnt/nas5/suhyeon/datasets/valAGE-Set'
        self.mask_datasets = '/mnt/nas5/suhyeon/datasets/valAGE-Set-Mask'
        self.image_path = '/mnt/nas5/suhyeon/datasets/valAGE-Set/0003.png'
        self.exp_name = 'baseline'
        self.output_dir = f'/mnt/nas5/suhyeon/projects/freq-loc/{self.exp_name}'

        # --- Model Configurations ---
        self.vae_model_name = "stabilityai/stable-diffusion-2-1"
        self.vae_subfolder = "vae"
        
        # --- Image Size Parameters ---
        self.vae_image_size = 512
        self.image_size = 256
        self.transform = transforms.Compose([
            transforms.Resize((self.image_size, self.image_size)),
            transforms.ToTensor(),
        ])

        # --- LocMark Core Parameters ---
        self.message_bits = 48
        self.margin = 1.0
        self.grid_size = 28
        self.mask_percentage = 0.3
        self.num_masks = 1
        self.seed = 42
        self.num_inference_steps = 100
        self.guidance_scale = 7.5

        # --- Optimization Parameters ---
        self.lr = 2.0
        self.steps = 500
        self.lambda_p = 0.0025 #0.025
        self.lambda_i = 0.005 #0.005
        self.feat_layer = 1

        # --- Robustness Parameters --- 
        self.eps0_std = [0.0, 0.8] # Latent noise
        
        # --- Demo/Evaluation Parameters ---
        self.batch_size = 1
        self.num_test_images = 1

        self.feature_dim = None
        if self.feat_layer == 0:
            self.feature_dim = 96
        elif self.feat_layer == 1:
            self.feature_dim = 192
        elif self.feat_layer == 2:
            self.feature_dim = 384
        elif self.feat_layer == 3:
            self.feature_dim = 768

In [15]:
def compute_psnr(a, b):
    mse = F.mse_loss(a, b).item()
    if mse == 0:
        return 100.0
    return 20 * torch.log10(1.0 / torch.sqrt(torch.tensor(mse)))

def calculate_iou(pred_mask, gt_mask):
    # Ensure masks are binary
    # pred_mask_bin = (pred_mask < 0).float()
    pred_mask_bin = torch.sigmoid(pred_mask)
    pred_mask_bin = (pred_mask_bin > 0.65).float() # Thresholding at 0.65
    gt_mask_bin = (gt_mask > 0).float() # Ground truth might not be 0/1

    save_image(pred_mask, "results/pred.png")
    save_image(pred_mask_bin, "results/pred_bin.png")
    save_image(gt_mask_bin, "results/gt.png")
    save_image(pred_mask_bin * gt_mask_bin, "results/intersection.png")
    save_image(pred_mask_bin + gt_mask_bin, "results/union.png")

    # Intersection and Union
    intersection = (pred_mask_bin * gt_mask_bin).sum()
    union = (pred_mask_bin + gt_mask_bin).sum() - intersection

    iou = intersection / (union + 1e-6) # Add epsilon to avoid division by zero
    return iou.item()

In [ ]:
# img_path = "/mnt/nas5/suhyeon/projects/freq-loc/secret_code/0002.png"
# img_path = "/mnt/nas5/suhyeon/projects/freq-loc/secret_code/analysis_dist_wm_step400.png"
# img_path = "/mnt/nas5/suhyeon/projects/freq-loc/baseline/20251120-152728/watermarked/0003.png"
# img_path = "/mnt/nas5/suhyeon/projects/locmark/20251120-185357/watermarked/0003.png"

seed = 45
proportion_masked = 0.3
trials = 5

In [ ]:
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-inpainting",
    # torch_dtype=torch.float16,
    cache_dir='/mnt/nas5/suhyeon/caches'
).to(device)

args = Params()
locmark = LocMark(args=args)

# secret_key = torch.load('./learned_directional_vector.pt')
# locmark.direction_vectors = torch.tensor(secret_key).to(args.device)
# print(locmark.direction_vectors)

torch.manual_seed(seed)
generator = torch.Generator(device=device).manual_seed(seed)
to_tensor = transforms.ToTensor()

# watermarked = load_img(img_path, transforms=args.transform)
# original = load_img('/mnt/nas5/suhyeon/datasets/valAGE-Set/0003.png', transforms=val_transforms)

image_files = sorted(glob.glob(os.path.join(args.image_datasets, "*.png")))[:5] # Limit to 5
mask_files = sorted(glob.glob(os.path.join(args.mask_datasets, "*.png")))[:5]   # Limit to 5

# original = F.interpolate(original, size=(512, 512), mode="bilinear", align_corners=False)
# watermarked = F.interpolate(watermarked, size=(512, 512), mode="bilinear", align_corners=False)

psnrs = []
ious = []
logits = []

for idx, (img_path, mask_path) in enumerate(zip(image_files, mask_files)):
    print(f"Processing pair {idx+1}: {os.path.basename(img_path)}")
    # mask = create_random_mask(watermarked, num_masks=1, mask_percentage=proportion_masked)

    original = load_image(img_path, transforms=val_transforms).to(device)
    mask = load_mask(mask_path, transforms=val_transforms).to(device)

    img_norm, min_norm, max_norm = norm_tensor(watermarked)
    img_edit_pil = pipe(prompt="", image=img_norm, mask_image=mask, generator=generator).images[0]
    img_edit = to_tensor(img_edit_pil)
    img_edit = img_edit.unsqueeze(0).to(device)

    img_edit = denorm_tensor(img_edit, min_norm, max_norm)  # [1, 3, H, W]
    # img_edit = img_edit * mask + watermarked * (1-mask)

    img_edit = F.interpolate(img_edit, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    decoded_batch = locmark.decode_watermark(img_edit)

    save_image(img_edit, "results/edited.png")
    # original = F.interpolate(original, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    # watermarked_224 = F.interpolate(watermarked, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    # mask_224 = F.interpolate(mask, size=(args.image_size, args.image_size), mode="bilinear", align_corners=False)
    psnrs.append(compute_psnr(watermarked, original))
    ious.append(calculate_iou(decoded_batch, 1-mask))
    logits.append(decoded_batch)
 
    print(f"PSNR: {psnrs[-1]:.2f}, IoU: {ious[-1]:.4f}")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]An error occurred while trying to fetch /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  57%|█████▋    | 4/7 [00:02<00:01,  1.85it/s]An error occurred while trying to fetch /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /mnt/nas5/suhyeon/caches/models--sd-legacy--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe 

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /opt/conda/envs/stableguard/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  6.00it/s]


PSNR: 31.01, IoU: 0.7510
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  5.99it/s]


...

PSNR: 31.01, IoU: 0.7320
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  5.98it/s]


PSNR: 31.01, IoU: 0.7073
Tensor normalized: min=0.0, max=1.0


100%|██████████| 50/50 [00:08<00:00,  5.98it/s]


PSNR: 31.01, IoU: 0.7336


In [18]:
print(f"## Average on {trials} trials ##")
print(f"PSNR (imperceptibility): {np.mean(psnrs):.2f} dB")
print(f"IoU (localization accuracy): {np.mean(ious):.4f}")

## Average on 5 trials ##
PSNR (imperceptibility): 31.01 dB
IoU (localization accuracy): 0.7298


In [19]:
# sig = torch.sigmoid(torch.cat(logits, dim=0)).cpu().numpy().flatten()

In [20]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 6))
# plt.hist(sig, bins=50, alpha=0.7)#, label='A: w/ L1 loss')
# plt.title('Logit Distribution Comparison')
# plt.xlabel('Logit Value')
# plt.ylabel('Frequency')
# plt.legend()
# plt.grid(True)
# plt.savefig('logits_comparison.png')
# # print("\nSaved logit distribution histogram to 'logit_histogram.png'")

In [21]:
# # logits_a = torch.load("logits_wo_loss.pt").cpu().numpy().flatten()
# logits_a = torch.load("logits_wo_loss.pt").cpu().numpy().flatten()
# logits_b = torch.load("logits_w_l1_loss.pt").cpu().numpy().flatten()
# logits_c = total_logits
# print(f"[A: w/o Add. Loss]Mean: {logits_a.mean():.2f}, Std: {logits_a.std():.2f}, Min: {logits_a.min():.2f}, Max: {logits_a.max():.2f}")
# print(f"[B: w/ L1 Loss] Mean: {logits_b.mean():.2f}, Std: {logits_b.std():.2f}, Min: {logits_b.min():.2f}, Max: {logits_b.max():.2f}")
# print(f"[B: w/ L1 Loss (Dual)] Mean: {logits_c.mean():.2f}, Std: {logits_c.std():.2f}, Min: {logits_c.min():.2f}, Max: {logits_c.max():.2f}")

# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 6))
# plt.hist(logits_a, bins=50, alpha=0.4, label='A: w/o L')
# plt.hist(logits_b, bins=50, alpha=0.4, label='B: w/ L')
# plt.hist(logits_c, bins=50, alpha=0.4, label='B: w/ L (Dual)')
# plt.title('Logit Distribution Comparison')
# plt.xlabel('Logit Value')
# plt.ylabel('Frequency')
# plt.legend()
# plt.grid(True)
# plt.savefig('logits_comparison.png')


In [22]:
# sig_a = torch.sigmoid(torch.load("logits_wo_loss.pt")).cpu().numpy().flatten()
# sig_b = torch.sigmoid(torch.load("logits_w_l1_loss.pt")).cpu().numpy().flatten()
# sig_c = torch.sigmoid(torch.load("logits_w_l1_loss_dual.pt")).cpu().numpy().flatten()
# print(f"[A: w/o Add. Loss]Mean: {sig_a.mean():.2f}, Std: {sig_a.std():.2f}, Min: {sig_a.min():.2f}, Max: {sig_a.max():.2f}")
# print(f"[B: w/ L1 Loss] Mean: {sig_b.mean():.2f}, Std: {sig_b.std():.2f}, Min: {sig_b.min():.2f}, Max: {sig_b.max():.2f}")
# print(f"[C: w/ L1 Loss (Dual)] Mean: {sig_c.mean():.2f}, Std: {sig_c.std():.2f}, Min: {sig_c.min():.2f}, Max: {sig_c.max():.2f}")

<!--  -->

In [23]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 6))
# plt.hist(sig_a, bins=50, alpha=0.7, label='A: w/o Add. Loss')
# plt.hist(sig_b, bins=50, alpha=0.7, label='B: w/ L1 Loss')
# plt.hist(sig_c, bins=50, alpha=0.7, label='B: w/ L1 Loss (Dual)')
# plt.title('Logit Distribution Comparison')
# plt.xlabel('Logit Value')
# plt.ylabel('Frequency')
# plt.legend()
# plt.grid(True)
# plt.savefig('logits_comparison.png')
# # print("\nSaved logit distribution histogram to 'logit_histogram.png'")

In [24]:
# 